# Chapter 9 — SciPy: distributions, integration, optimization

**Goal:** use `scipy.stats` as a toolbox of probability distributions (PDFs, CDFs, quantiles, sampling),
integrate with `scipy.integrate.quad`, and minimize functions with `scipy.optimize`, including fitting a model by maximum likelihood.

**Where you'll meet this in MLMA Lab 1:** the *SciPy* section (`minimize`, `scipy.linalg`, `csr_matrix`, `norm.cdf`,
`savemat`/`loadmat`). In the probability labs `scipy.stats` is the main tool.

In [ ]:
import numpy as np
from scipy import stats

## 9.1 Distribution objects in `scipy.stats`

Every distribution has the same methods:

| method | meaning | continuous example |
|---|---|---|
| `pdf(x)` / `pmf(k)` | density / probability mass | `stats.norm.pdf(0)` |
| `cdf(x)` | $P(X \le x)$ | `stats.norm.cdf(1.96)` ≈ 0.975 |
| `sf(x)` | survival $P(X > x) = 1 - \text{cdf}$ (more accurate in the tails) | |
| `ppf(q)` | quantile, the inverse of the CDF | `stats.norm.ppf(0.975)` ≈ 1.96 |
| `rvs(size, random_state)` | random samples | |
| `mean()`, `var()`, `std()`, `interval(conf)` | summaries | |

**Parameters are `loc` and `scale`** for continuous distributions, and they don't always match textbook notation:

| textbook | SciPy |
|---|---|
| $N(\mu, \sigma^2)$ | `stats.norm(loc=mu, scale=sigma)` (scale is the **std**, not the variance!) |
| $\text{Uniform}(a, b)$ | `stats.uniform(loc=a, scale=b - a)` |
| $\text{Exp}(\lambda)$ (rate) | `stats.expon(scale=1 / lam)` |
| $\text{Binomial}(n, p)$ | `stats.binom(n, p)` |
| $\text{Poisson}(\lambda)$ | `stats.poisson(lam)` |

Calling the distribution with its parameters gives a **frozen** distribution, which you can store and reuse:

In [ ]:
print(stats.norm.cdf(0.5))                  # the line from the MLMA lab

X = stats.norm(loc=170, scale=10)           # frozen: heights ~ N(170, 10^2)
print(X.mean(), X.std(), X.cdf(180), X.ppf(0.5), X.interval(0.95))

B = stats.binom(10, 0.3)
print(B.pmf(3), B.cdf(3), B.mean(), B.var())
print(B.rvs(size=5, random_state=0))

### Discrete off-by-one trap

For a discrete variable $P(X \ge k) = 1 - P(X \le k-1)$ = `sf(k - 1)`, **not** `sf(k)`. Write it out every time.

## 9.2 Integration with `quad`

`quad(f, a, b)` returns a tuple `(value, error_estimate)`. Use `np.inf` for infinite limits.

In [ ]:
from scipy.integrate import quad

value, err = quad(stats.norm.pdf, -np.inf, np.inf)
print(value, err)
mean_value, _ = quad(lambda x: x * stats.expon(scale=2).pdf(x), 0, np.inf)   # E[X] = 2
print(mean_value)

## 9.3 Optimization

- `minimize_scalar(f)`: one variable.
- `minimize(f, x0)`: many variables, starting from guess `x0`. It returns a result object: `res.x` (the argmin),
  `res.fun` (the minimum value) and `res.success`.
- To **maximize** something, minimize its negative. This is how maximum likelihood is done: minimize the *negative log-likelihood*.

These methods find a **local** minimum, so a different `x0` may give a different answer.

In [ ]:
from scipy.optimize import minimize, minimize_scalar

fun = lambda x: x ** 2 + 10 * np.sin(x)
res = minimize(fun, x0=0)
print(res.x, res.fun, res.success)
print(minimize(fun, x0=5).x)            # different start, different local minimum

## 9.4 Other submodules you'll see

- `scipy.linalg`: like `np.linalg`, with more functions (`inv`, `det`, `lu`, `cholesky`, …)
- `scipy.sparse`: matrices that are mostly zeros, e.g. `csr_matrix`
- `scipy.io`: `savemat` / `loadmat` for MATLAB `.mat` files
- `scipy.special`: `comb`, `gamma`, `logsumexp`, `expit` (sigmoid), …

---
# Exercises

### Exercise 9.1: The normal distribution
IQ scores are modelled as $N(100, 15^2)$. Using `stats.norm` with `loc` and `scale`:
- `p_above_130`: $P(X > 130)$ (use `sf`)
- `p_between`: $P(85 < X < 115)$
- `q99`: the score that 99% of people are below
- `dens_100`: the density at 100

In [ ]:
p_above_130 = ...
p_between = ...
q99 = ...
dens_100 = ...

In [ ]:
assert np.isclose(p_above_130, 0.022750131948179195)
assert np.isclose(p_between, 0.6826894921370859)
assert np.isclose(q99, 134.89521811)
assert np.isclose(dens_100, 1 / (15 * np.sqrt(2 * np.pi)))
print("Exercise 9.1 passed")

### Exercise 9.2: Frozen distributions and summaries
Create the frozen distribution `T`, the waiting time ~ Exponential with **rate** $\lambda = 0.5$ per minute.
- `T_mean`, `T_var`
- `p_wait_long`: $P(T > 4)$
- `ci`: the central 90% interval (`T.interval(0.9)`)

**Predict first:** what should `T_mean` be?

In [ ]:
lam = 0.5
T = ...
T_mean = ...
T_var = ...
p_wait_long = ...
ci = ...

In [ ]:
assert np.isclose(T_mean, 2) and np.isclose(T_var, 4), "remember: scale = 1 / rate"
assert np.isclose(p_wait_long, np.exp(-2))
assert np.isclose(ci[0], -2 * np.log(0.95)) and np.isclose(ci[1], -2 * np.log(0.05))
print("Exercise 9.2 passed")

### Exercise 9.3: Binomial and the off-by-one trap
A test has 20 multiple-choice questions with 4 options each. Someone guesses every answer. $X$ = number correct.
- `p_exactly_5`: $P(X = 5)$
- `p_at_most_3`: $P(X \le 3)$
- `p_at_least_10`: $P(X \ge 10)$. Careful!
- `p_pass_check`: the same value as `p_at_least_10`, computed by summing `pmf` over `k = 10..20` (to confirm you got the trap right)

In [ ]:
p_exactly_5 = ...
p_at_most_3 = ...
p_at_least_10 = ...
p_pass_check = ...

In [ ]:
assert np.isclose(p_exactly_5, 0.2023311519)
assert np.isclose(p_at_most_3, 0.2251560)
assert np.isclose(p_pass_check, p_at_least_10), "your two computations disagree"
assert np.isclose(p_at_least_10, 0.0138644), "P(X >= 10) = sf(9)"
print("Exercise 9.3 passed")

### Exercise 9.4: A whole PMF as an array
Calls to a help desk arrive as Poisson with mean 4 per hour.
- `ks`: `np.arange(0, 16)`
- `pmf`: the Poisson PMF evaluated at **all** of `ks` in one call
- `mode_k`: the most likely number of calls (use `argmax`)
- `tail`: the probability mass **not** covered by `ks` (`1 - pmf.sum()`, or `sf(15)`)

In [ ]:
ks = ...
pmf = ...
mode_k = ...
tail = ...

In [ ]:
assert pmf.shape == (16,)
assert mode_k in (3, 4), "for integer lambda the PMF has a tie at lambda-1 and lambda"
assert np.isclose(tail, stats.poisson(4).sf(15))
print("Exercise 9.4 passed")

### Exercise 9.5: Sampling and checking against theory
Draw `samples`, 50 000 values from a Gamma distribution with shape `a=3` and `scale=2`, using `.rvs(size=..., random_state=0)`.
Then compute `sample_mean` and `sample_var` (unbiased: `ddof=1`) and compare them with the theoretical `G.mean()` and `G.var()`.

In [ ]:
G = stats.gamma(a=3, scale=2)
samples = ...
sample_mean = ...
sample_var = ...
print(sample_mean, G.mean(), sample_var, G.var())

In [ ]:
assert samples.shape == (50_000,)
assert abs(sample_mean - 6) < 0.05 and abs(sample_var - 12) < 0.3
print("Exercise 9.5 passed")

### Exercise 9.6: Parametrization traps
Build these frozen distributions with the right `loc`/`scale`:
- `U`: uniform on $[2, 5]$
- `N`: normal with mean 1 and **variance** 4
- `E`: exponential with rate 3

In [ ]:
U = ...
N = ...
E = ...

In [ ]:
assert np.isclose(U.cdf(2), 0) and np.isclose(U.cdf(5), 1) and np.isclose(U.mean(), 3.5)
assert np.isclose(N.mean(), 1) and np.isclose(N.var(), 4), "scale is the standard deviation"
assert np.isclose(E.mean(), 1 / 3)
print("Exercise 9.6 passed")

### Exercise 9.7: Integration with `quad`
Let $f(x) = 6x(1 - x)$ on $[0, 1]$ (a Beta(2, 2) density).
- `total`: $\int_0^1 f$. Is it a valid density?
- `EX`: $E[X] = \int_0^1 x f(x)\,dx$
- `EX2`: $E[X^2]$, then `VarX = EX2 - EX ** 2`
- Compare `VarX` with `stats.beta(2, 2).var()`.

Remember that `quad` returns `(value, error)`.

In [ ]:
from scipy.integrate import quad
f = lambda x: 6 * x * (1 - x)
total = ...
EX = ...
EX2 = ...
VarX = ...

In [ ]:
assert np.isclose(total, 1) and np.isclose(EX, 0.5)
assert np.isclose(VarX, stats.beta(2, 2).var())
print("Exercise 9.7 passed")

### Exercise 9.8: Minimization and local minima
For $g(x) = x^2 + 10 \sin(x)$ (the MLMA lab's function):
- `res_a`: `minimize(g, x0=0)`
- `res_b`: `minimize(g, x0=5)`
- `global_x`: the x of whichever result has the **lower** function value (compare `res_a.fun` and `res_b.fun`)

Then try `minimize_scalar(g)` and see which minimum it finds.

In [ ]:
from scipy.optimize import minimize, minimize_scalar
g = lambda x: x ** 2 + 10 * np.sin(x)
res_a = ...
res_b = ...
global_x = ...
print(res_a.x, res_b.x, global_x)

In [ ]:
assert np.isclose(float(np.squeeze(global_x)), -1.30644, atol=1e-3)
assert res_b.fun > res_a.fun, "x0=5 should end up in the other, worse, local minimum"
print("Exercise 9.8 passed")

### Exercise 9.9: Maximum likelihood with `minimize`
You observe `data` and assume it's $N(\mu, \sigma^2)$. Fit $\mu$ and $\sigma$ by **minimizing the negative log-likelihood**:

$$\text{NLL}(\mu, \sigma) = -\sum_i \log f(x_i \mid \mu, \sigma)$$

1. Write `nll(params)` where `params = [mu, sigma]`. Use `stats.norm.logpdf(data, loc=mu, scale=sigma).sum()`.
   Return `np.inf` if `sigma <= 0`.
2. `res = minimize(nll, x0=[0, 1], method="Nelder-Mead")`
3. `mu_hat, sigma_hat = res.x`

**Predict:** which simple NumPy statistics should these match? (The check compares against them.)

In [ ]:
data = stats.norm(loc=3, scale=1.5).rvs(size=500, random_state=42)

def nll(params):
    mu, sigma = params
    pass  # YOUR CODE HERE

res = ...
mu_hat, sigma_hat = ..., ...
print(mu_hat, sigma_hat)

In [ ]:
assert np.isclose(mu_hat, data.mean(), atol=1e-3)
assert np.isclose(sigma_hat, data.std(ddof=0), atol=1e-3), "the MLE of sigma uses ddof=0"
print("Exercise 9.9 passed")

### Exercise 9.10: Describing a sample
For `sample` below, use `scipy.stats`:
- `desc = stats.describe(sample)`, then look at the printout
- `skewness`: `stats.skew(sample)`. A right-skewed sample should give a positive value.
- `z_scores`: `stats.zscore(sample)`
- `pct_rank`: the percentile rank of the value 5 within the sample (`stats.percentileofscore`)

In [ ]:
sample = stats.expon(scale=3).rvs(size=2000, random_state=7)
desc = ...
skewness = ...
z_scores = ...
pct_rank = ...
print(desc)

In [ ]:
assert desc.nobs == 2000
assert skewness > 1, "the exponential distribution is strongly right-skewed (theory: 2)"
assert np.isclose(z_scores.mean(), 0) and np.isclose(z_scores.std(), 1)
assert np.isclose(pct_rank, (sample <= 5).mean() * 100, atol=0.1)
print("Exercise 9.10 passed")

### Exercise 9.11: `scipy.linalg` and sparse matrices
- `A_inv`: inverse of `A` with `scipy.linalg.inv`; verify that `A @ A_inv` is the identity
- `S`: `csr_matrix(dense)`
- `n_stored`: the number of stored non-zeros (`S.nnz`)
- `back`: `S` converted back to a dense NumPy array (`.toarray()`)

In [ ]:
from scipy import linalg
from scipy.sparse import csr_matrix
A = np.array([[4.0, 7.0], [2.0, 6.0]])
dense = np.array([[1, 2, 0], [0, 0, 3], [4, 0, 5]])
A_inv = ...
S = ...
n_stored = ...
back = ...

In [ ]:
assert np.allclose(A @ A_inv, np.eye(2))
assert n_stored == 5 and np.array_equal(back, dense)
print("Exercise 9.11 passed")

### Exercise 9.12: `.mat` files
Save the dict `{"x": x, "label": label}` to `"primer_data.mat"` with `savemat`, load it back into `loaded`,
then delete the file.

Look at `loaded["x"].shape`. MATLAB has no 1-D arrays, so a 1-D array comes back as a **2-D** `(1, n)` array.
Set `x_back` to a 1-D version (`.ravel()`).

In [ ]:
from pathlib import Path
from scipy.io import savemat, loadmat
x = np.linspace(0, 1, 5)
label = "trial"
loaded = ...
x_back = ...

In [ ]:
assert loaded["x"].shape == (1, 5)
assert np.allclose(x_back, x) and x_back.shape == (5,)
assert not Path("primer_data.mat").exists()
print("Exercise 9.12 passed")

---
**Chapter 9 done.** Next: Chapter 10, pandas.